In [ ]:
from flask import Flask, render_template, request
from PIL import Image
import numpy as np
import os

app = Flask(__name__)

UPLOAD_FOLDER = "static/uploads"
app.config["UPLOAD_FOLDER"] = UPLOAD_FOLDER

os.makedirs(UPLOAD_FOLDER, exist_ok=True)


def get_top_colors(image, number_of_colors=10):
    # Convert image to RGB
    image = image.convert("RGB")

    # Resize the image to make processing faster
    image.thumbnail((300, 300))

    # Convert image into a NumPy array
    image_array = np.array(image)

    # Reshape the image into a list of RGB pixels
    pixels = image_array.reshape(-1, 3)

    # Count unique colours
    colors, counts = np.unique(pixels, axis=0, return_counts=True)

    # Sort colours according to frequency
    sorted_indices = np.argsort(counts)[::-1]

    top_colors = []

    for index in sorted_indices[:number_of_colors]:
        rgb = colors[index]
        count = counts[index]

        hex_color = "#{:02x}{:02x}{:02x}".format(
            rgb[0], rgb[1], rgb[2]
        )

        top_colors.append({
            "hex": hex_color.upper(),
            "rgb": tuple(rgb),
            "count": int(count)
        })

    return top_colors


@app.route("/", methods=["GET", "POST"])
def index():
    colors = []
    image_path = None

    if request.method == "POST":

        if "image" not in request.files:
            return render_template(
                "index.html",
                error="Please select an image."
            )

        file = request.files["image"]

        if file.filename == "":
            return render_template(
                "index.html",
                error="Please select an image."
            )

        try:
            image = Image.open(file)

            # Save uploaded image
            filename = "uploaded_image.png"
            save_path = os.path.join(
                app.config["UPLOAD_FOLDER"],
                filename
            )

            image.save(save_path)

            image_path = "/" + save_path.replace("\\", "/")

            # Find top 10 colours
            colors = get_top_colors(image, 10)

        except Exception:
            return render_template(
                "index.html",
                error="There was a problem processing the image."
            )

    return render_template(
        "index.html",
        colors=colors,
        image_path=image_path
    )


if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
